# Practical Python OOP: AIML, Security, and Full-Stack Practice

A progressive, test-driven workbook of 50 practical OOP challenges, from fundamentals to advanced design, covering AIML, defensive security, and full-stack application development. Exercises use Python's standard library.

## How to work

1. Read a challenge and implement it in its **Your code** cell.
2. Run that cell, then run its **Check your work** cell. `PASS` means every listed check passed; failed checks name the behavior to revisit. Exceptions are left visible so implementation errors are diagnosable.
3. Work in order where possible. Challenges are independent unless they explicitly name a prior concept.
4. The complete answer key is at the very end. Try the challenges before scrolling there.

Run the helper cell near the top once. No packages need to be installed.

In [ ]:
import hashlib
import hmac
import json
import math
import random
import secrets
import time
from pathlib import Path

def check_exercise(title, checks):
    failed = [label for label, passed in checks if not passed]
    if failed:
        print(f"NOT YET: {title} ({len(failed)}/{len(checks)} checks failed)")
        for label in failed:
            print(f"  - {label}")
        return False
    print(f"PASS: {title} ({len(checks)} checks)")
    return True


def raises(expected_exception, action):
    try:
        action()
    except expected_exception:
        return True
    return False


## 01 · Classes, objects, attributes, and methods

**Beginner · OOP foundations**

**Problem.** Create `Book(title, pages)`. `label()` returns `"<title> (<pages> pages)"`. Different objects must keep independent data.

Implement the requested behavior, then run the check cell.

In [ ]:
class Book:
    def __init__(self, title, pages):
        raise NotImplementedError("Initialize this book")

    def label(self):
        raise NotImplementedError("Return the requested label")


In [ ]:
first = Book("Dune", 412)
second = Book("Kindred", 264)
check_exercise("01 · Book", [
    ("formatted label", first.label() == "Dune (412 pages)"),
    ("independent titles", second.title == "Kindred"),
    ("independent page counts", second.pages == 264),
])


## 02 · Encapsulation, properties, and validated state

**Beginner · Account domain model**

**Problem.** Implement `BankAccount` with a read-only `balance` property. Reject negative opening balances, non-positive deposits/withdrawals, and overdrafts with `ValueError`.

Implement the requested behavior, then run the check cell.

In [ ]:
class BankAccount:
    def __init__(self, owner, opening_balance=0):
        raise NotImplementedError("Validate and initialize the account")

    @property
    def balance(self):
        raise NotImplementedError("Expose the current balance")

    def deposit(self, amount):
        raise NotImplementedError("Deposit a positive amount")

    def withdraw(self, amount):
        raise NotImplementedError("Withdraw without allowing overdrafts")


In [ ]:
account = BankAccount("Amina", 100)
account.deposit(25)
account.withdraw(40)
check_exercise("02 · BankAccount", [
    ("balance after transactions", account.balance == 85),
    ("balance is read-only", raises(AttributeError, lambda: setattr(account, "balance", 500))),
    ("zero deposit rejected", raises(ValueError, lambda: account.deposit(0))),
    ("overdraft rejected", raises(ValueError, lambda: account.withdraw(86))),
    ("negative opening balance rejected", raises(ValueError, lambda: BankAccount("Bad", -1))),
])


## 03 · Class state, class methods, and static methods

**Beginner → intermediate · ML job tracking**

**Problem.** `TrainingRun` stores a per-object `model_name` and tracks created objects in a class attribute. Add class method `count()` and static method `valid_epochs(value)` (positive int, not bool). The checks are safe to rerun.

Implement the requested behavior, then run the check cell.

In [ ]:
class TrainingRun:
    created_count = 0

    def __init__(self, model_name):
        raise NotImplementedError("Store the name and update the class count")

    @classmethod
    def count(cls):
        raise NotImplementedError("Return the created count")

    @staticmethod
    def valid_epochs(value):
        raise NotImplementedError("Check for a positive integer, excluding bool")


In [ ]:
before = TrainingRun.count()
first = TrainingRun("linear")
second = TrainingRun("forest")
check_exercise("03 · TrainingRun", [
    ("class count increases per object", TrainingRun.count() == before + 2),
    ("instance names are independent", (first.model_name, second.model_name) == ("linear", "forest")),
    ("positive epochs accepted", TrainingRun.valid_epochs(5)),
    ("zero rejected", not TrainingRun.valid_epochs(0)),
    ("bool rejected", not TrainingRun.valid_epochs(True)),
    ("float rejected", not TrainingRun.valid_epochs(1.5)),
])


## 04 · Dunder methods and value objects

**Intermediate · AIML result object**

**Problem.** Implement `PredictionResult(label, confidence)`. Confidence is in `[0, 1]`; `str()` formats as `"spam: 92%"`. Equality compares both fields.

Implement the requested behavior, then run the check cell.

In [ ]:
class PredictionResult:
    def __init__(self, label, confidence):
        raise NotImplementedError("Validate and store the result")

    def __str__(self):
        raise NotImplementedError("Format the label and percentage")

    def __eq__(self, other):
        raise NotImplementedError("Compare both fields")


In [ ]:
prediction = PredictionResult("spam", 0.92)
check_exercise("04 · PredictionResult", [
    ("readable string form", str(prediction) == "spam: 92%"),
    ("matching values compare equal", prediction == PredictionResult("spam", 0.92)),
    ("different values compare unequal", prediction != PredictionResult("ham", 0.92)),
    ("invalid confidence rejected", raises(ValueError, lambda: PredictionResult("spam", 1.1))),
])


## 05 · Inheritance, overriding, and super()

**Intermediate · Reusable AIML models**

**Problem.** Create `Predictor` and subclass `LinearPredictor`. Initialize the base name with `super()`. Prediction is the weighted sum plus bias; reject feature/weight length mismatches.

Implement the requested behavior, then run the check cell.

In [ ]:
class Predictor:
    def __init__(self, name):
        self.name = name

    def predict(self, features):
        raise NotImplementedError("Subclasses implement prediction")


class LinearPredictor(Predictor):
    def __init__(self, weights, bias=0):
        raise NotImplementedError("Initialize base and model parameters")

    def predict(self, features):
        raise NotImplementedError("Compute weighted sum plus bias")


In [ ]:
model = LinearPredictor([2, 3], bias=1)
check_exercise("05 · LinearPredictor", [
    ("base name initialized", model.name == "linear"),
    ("weighted prediction", model.predict([4, 5]) == 24),
    ("length mismatch rejected", raises(ValueError, lambda: model.predict([4]))),
])


## 06 · Polymorphism and duck typing

**Intermediate · Model ensembles**

**Problem.** Write `batch_predict(models, features)`. Call each object's `predict(features)` and return results in order without checking concrete model types.

Implement the requested behavior, then run the check cell.

In [ ]:
def batch_predict(models, features):
    raise NotImplementedError("Call predict on each model")


In [ ]:
class AddModel:
    def predict(self, values):
        return sum(values)

class MaxModel:
    def predict(self, values):
        return max(values)

check_exercise("06 · Polymorphic prediction", [
    ("different classes share one interface", batch_predict([AddModel(), MaxModel()], [2, 5, 3]) == [10, 5]),
    ("empty model list", batch_predict([], [1]) == []),
])


## 07 · Composition and preprocessing pipelines

**Intermediate · Data preparation**

**Problem.** Implement `FeaturePipeline(transformers)`. `transform(values)` passes each result to the next transformer in order. Each transformer has a `transform` method.

Implement the requested behavior, then run the check cell.

In [ ]:
class FeaturePipeline:
    def __init__(self, transformers):
        raise NotImplementedError("Store the composed transformers")

    def transform(self, values):
        raise NotImplementedError("Apply transforms in order")


In [ ]:
class AddOffset:
    def transform(self, values):
        return [value + 1 for value in values]

class DoubleValues:
    def transform(self, values):
        return [value * 2 for value in values]

check_exercise("07 · FeaturePipeline", [
    ("composition order", FeaturePipeline([AddOffset(), DoubleValues()]).transform([1, 3]) == [4, 8]),
    ("empty pipeline", FeaturePipeline([]).transform([2, 4]) == [2, 4]),
])


## 08 · Abstract base classes and contracts

**Intermediate · Backend persistence**

**Problem.** Define abstract `Storage.save` and `Storage.load`. Implement `InMemoryStorage`; missing keys return `None`. The abstract base must not be instantiable.

Implement the requested behavior, then run the check cell.

In [ ]:
from abc import ABC, abstractmethod

class Storage(ABC):
    @abstractmethod
    def save(self, key, value):
        raise NotImplementedError

    @abstractmethod
    def load(self, key):
        raise NotImplementedError


class InMemoryStorage(Storage):
    def __init__(self):
        raise NotImplementedError("Initialize storage")

    def save(self, key, value):
        raise NotImplementedError("Save the value")

    def load(self, key):
        raise NotImplementedError("Load a value or return None")


In [ ]:
storage = InMemoryStorage()
storage.save("model:v1", {"accuracy": 0.91})
check_exercise("08 · Storage", [
    ("saved value can be loaded", storage.load("model:v1") == {"accuracy": 0.91}),
    ("missing key returns None", storage.load("missing") is None),
    ("abstract base cannot be instantiated", raises(TypeError, lambda: Storage())),
])


## 09 · Dataclasses and custom validation errors

**Intermediate · Typed ML data**

**Problem.** Create frozen dataclass `FeatureVector(names, values)` and custom exception `DataValidationError`. Reject mismatched vector lengths. Implement `DatasetBatch(features, labels)` to reject empty data, mismatched row counts, or empty rows; `len(batch)` returns row count.

Implement the requested behavior, then run the check cell.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class FeatureVector:
    names: tuple
    values: tuple

    def __post_init__(self):
        raise NotImplementedError("Validate vector lengths")


class DataValidationError(ValueError):
    pass


class DatasetBatch:
    def __init__(self, features, labels):
        raise NotImplementedError("Validate and store the batch")

    def __len__(self):
        raise NotImplementedError("Return the row count")


In [ ]:
vector = FeatureVector(("age", "score"), (30, 0.8))
batch = DatasetBatch([[1, 2], [3, 4]], [0, 1])
check_exercise("09 · Validated data", [
    ("dataclass equality", vector == FeatureVector(("age", "score"), (30, 0.8))),
    ("mismatched feature vector rejected", raises(ValueError, lambda: FeatureVector(("age",), (30, 0.8)))),
    ("dataclass is frozen", raises(AttributeError, lambda: setattr(vector, "values", (1,)))),
    ("dataset length counts rows", len(batch) == 2),
    ("mismatched counts rejected", raises(DataValidationError, lambda: DatasetBatch([[1]], []))),
    ("empty row rejected", raises(DataValidationError, lambda: DatasetBatch([[]], [1]))),
    ("empty data rejected", raises(DataValidationError, lambda: DatasetBatch([], []))),
])


## 10 · Mixins and method resolution order

**Advanced · Reusable model metadata**

**Problem.** Create `TimestampMixin.timestamp()` and `JSONExportMixin.to_json()`. Implement `TrainingCard(TimestampMixin, JSONExportMixin)` with `model_name` and `created_at` fields.

Implement the requested behavior, then run the check cell.

In [ ]:
import json

class TimestampMixin:
    def timestamp(self):
        raise NotImplementedError("Return created_at")

class JSONExportMixin:
    def to_json(self):
        raise NotImplementedError("Serialize instance data as sorted JSON")

class TrainingCard(TimestampMixin, JSONExportMixin):
    def __init__(self, model_name, created_at):
        raise NotImplementedError("Store card fields")


In [ ]:
import json
card = TrainingCard("xgboost", "2026-01-01")
check_exercise("10 · TrainingCard mixins", [
    ("timestamp method", card.timestamp() == "2026-01-01"),
    ("JSON serialization", json.loads(card.to_json()) == {"created_at": "2026-01-01", "model_name": "xgboost"}),
    ("MRO follows declared order", TrainingCard.mro()[1:3] == [TimestampMixin, JSONExportMixin]),
])


## 11 · Descriptors and reusable attribute validation

**Advanced · Configuration objects**

**Problem.** Implement descriptor `PositiveFloat` and use it for `TrainingConfig.learning_rate`. Reject booleans, non-numbers, and non-positive values with `ValueError`.

Implement the requested behavior, then run the check cell.

In [ ]:
class PositiveFloat:
    def __set_name__(self, owner, name):
        raise NotImplementedError("Remember the attribute name")

    def __get__(self, instance, owner):
        raise NotImplementedError("Read the instance value")

    def __set__(self, instance, value):
        raise NotImplementedError("Validate and store a positive number")


class TrainingConfig:
    learning_rate = PositiveFloat()

    def __init__(self, learning_rate):
        self.learning_rate = learning_rate


In [ ]:
config = TrainingConfig(0.01)
check_exercise("11 · PositiveFloat", [
    ("valid value is stored", config.learning_rate == 0.01),
    ("zero rejected", raises(ValueError, lambda: setattr(config, "learning_rate", 0))),
    ("text rejected", raises(ValueError, lambda: setattr(config, "learning_rate", "fast"))),
    ("bool rejected", raises(ValueError, lambda: setattr(config, "learning_rate", True))),
])


## 12 · Protocols and structural typing

**Advanced · Typed AIML interfaces**

**Problem.** Define `PredictorProtocol.predict(features) -> float` and `run_prediction(model, features)`. Use the protocol as a type hint and accept any compatible object without inheritance.

Implement the requested behavior, then run the check cell.

In [ ]:
from typing import Protocol

class PredictorProtocol(Protocol):
    def predict(self, features: list[float]) -> float:
        ...

def run_prediction(model: PredictorProtocol, features: list[float]) -> float:
    raise NotImplementedError("Delegate prediction")


In [ ]:
class TinyModel:
    def predict(self, features):
        return sum(features) / len(features)

check_exercise("12 · Predictor protocol", [
    ("compatible object works without inheritance", run_prediction(TinyModel(), [2.0, 4.0]) == 3.0),
])


## 13 · Repository and service layers

**Advanced · Full-stack backend domain logic**

**Problem.** Implement in-memory `UserRepository` and `UserService`. Normalize email to lowercase, trim name, reject empty values and duplicates with `ValueError`, and return `{"email": ..., "name": ...}`. Keep storage behind the repository.

Implement the requested behavior, then run the check cell.

In [ ]:
class UserRepository:
    def __init__(self):
        self._users = {}

    def find_by_email(self, email):
        raise NotImplementedError("Look up a normalized email")

    def add(self, user):
        raise NotImplementedError("Store the user")


class UserService:
    def __init__(self, repository):
        self.repository = repository

    def register(self, email, name):
        raise NotImplementedError("Validate, normalize, and register")


In [ ]:
service = UserService(UserRepository())
user = service.register("  LEARNER@example.com ", "  Sam  ")
check_exercise("13 · User service", [
    ("email and name normalized", user == {"email": "learner@example.com", "name": "Sam"}),
    ("repository lookup", service.repository.find_by_email("learner@example.com") == user),
    ("duplicate email rejected", raises(ValueError, lambda: service.register("learner@example.com", "Other"))),
    ("empty name rejected", raises(ValueError, lambda: service.register("new@example.com", "  "))),
])


## 14 · Controller layer and API response shape

**Advanced · Full-stack HTTP boundary**

**Problem.** Implement `UserController.create(payload)` around an injected service. Return status 201 and the user on success; map service `ValueError` to status 400 and an error message. Keep persistence logic out of the controller.

Implement the requested behavior, then run the check cell.

In [ ]:
class UserController:
    def __init__(self, user_service):
        self.user_service = user_service

    def create(self, payload):
        raise NotImplementedError("Call the service and map its response")


In [ ]:
controller = UserController(UserService(UserRepository()))
created = controller.create({"email": "api@example.com", "name": "API User"})
invalid = controller.create({"email": "api@example.com", "name": "Duplicate"})
check_exercise("14 · UserController", [
    ("success uses status 201", created == {"status": 201, "body": {"email": "api@example.com", "name": "API User"}}),
    ("validation maps to status 400", invalid["status"] == 400 and "error" in invalid["body"]),
])


## 15 · AIML serving, composition, and dependency injection

**Advanced · Preprocessor + model + decision policy**

**Problem.** Implement `ServingService(preprocessor, model, threshold=0.5)`. Transform raw features, call `model.predict_proba`, and return `{"label": 0 or 1, "probability": positive_probability}`. Label is 1 at or above threshold; reject thresholds outside `[0, 1]`.

Implement the requested behavior, then run the check cell.

In [ ]:
class ServingService:
    def __init__(self, preprocessor, model, threshold=0.5):
        raise NotImplementedError("Validate and store injected dependencies")

    def predict(self, raw_features):
        raise NotImplementedError("Preprocess, predict, and apply threshold")


In [ ]:
class Shift:
    def transform(self, values):
        return [value + 10 for value in values]

class ThresholdModel:
    def predict_proba(self, values):
        positive = 0.8 if values[0] > 10 else 0.2
        return [1 - positive, positive]

service = ServingService(Shift(), ThresholdModel(), threshold=0.7)
check_exercise("15 · AIML ServingService", [
    ("preprocessor runs before model", service.predict([1]) == {"label": 1, "probability": 0.8}),
    ("threshold controls label", ServingService(Shift(), ThresholdModel(), 0.9).predict([1])["label"] == 0),
    ("invalid threshold rejected", raises(ValueError, lambda: ServingService(Shift(), ThresholdModel(), 1.1))),
])


## 16 · Class-level registries and factories

**Advanced · Model/plugin registration**

**Problem.** `ModelRegistry` stores zero-argument factories by name. `register` and `create` are class methods. `create` returns a fresh object each time and raises `KeyError` for unknown names.

Implement the requested behavior, then run the check cell.

In [ ]:
class ModelRegistry:
    _factories = {}

    @classmethod
    def register(cls, name, factory):
        raise NotImplementedError("Register a factory")

    @classmethod
    def create(cls, name):
        raise NotImplementedError("Construct a fresh object or raise KeyError")


In [ ]:
class DemoModel:
    pass

ModelRegistry.register("demo-workbook", DemoModel)
first = ModelRegistry.create("demo-workbook")
second = ModelRegistry.create("demo-workbook")
check_exercise("16 · ModelRegistry", [
    ("factory returns requested type", isinstance(first, DemoModel)),
    ("factory returns a fresh instance", first is not second),
    ("unknown model rejected", raises(KeyError, lambda: ModelRegistry.create("unknown-workbook-model"))),
])


## 17 · Context managers and transactional cleanup

**Advanced · Safe updates**

**Problem.** Implement `AtomicList` as a context manager. Save a snapshot on entry. If the `with` block raises, restore the original list and propagate the exception; otherwise keep changes.

Implement the requested behavior, then run the check cell.

In [ ]:
class AtomicList:
    def __init__(self, values):
        self.values = list(values)

    def __enter__(self):
        raise NotImplementedError("Save a snapshot and return self")

    def __exit__(self, exception_type, exception, traceback):
        raise NotImplementedError("Restore on failure and propagate exceptions")


In [ ]:
items = AtomicList([1, 2])
def fail_inside_transaction():
    with items as transaction:
        transaction.values.append(3)
        raise RuntimeError("rollback")

propagated = raises(RuntimeError, fail_inside_transaction)
with items as transaction:
    transaction.values.append(4)
check_exercise("17 · AtomicList", [
    ("exception propagates", propagated),
    ("failed block rolls back", items.values == [1, 2, 4]),
    ("successful block keeps changes", items.values[-1] == 4),
])


# AIML practical challenge set

These six interview-style exercises cover end-to-end machine-learning work using only the Python standard library. Implement your answer in its code cell and run the check cell. Worked answers follow all exercises.

## AIML 01 · Train/test split without leakage

**Dataset preparation**

**Problem.** Implement `split_indices(row_count, test_ratio=0.25, seed=0)`. Return shuffled `(train_indices, test_indices)` with no overlap, covering every input row once. The same seed must give the same split. Reject fewer than two rows and ratios outside `(0, 1)` with `ValueError`.

Write your solution in the next cell, then run its check cell.

In [ ]:
def split_indices(row_count, test_ratio=0.25, seed=0):
    raise NotImplementedError("Create reproducible, disjoint train and test indices")


In [ ]:
train, test = split_indices(20, test_ratio=0.25, seed=7)
check_exercise("AIML 01 · Train/test split", [
    ("split is reproducible", (train, test) == split_indices(20, 0.25, 7)),
    ("all rows appear once", sorted(train + test) == list(range(20))),
    ("train and test do not overlap", set(train).isdisjoint(test)),
    ("test size follows ratio", len(test) == 5),
    ("single-row dataset rejected", raises(ValueError, lambda: split_indices(1))),
    ("invalid ratio rejected", raises(ValueError, lambda: split_indices(10, 1.0))),
])


## AIML 02 · Confusion matrix and classification metrics

**Classification evaluation**

**Problem.** Implement `classification_metrics(y_true, y_pred)` for binary labels. Return `tp`, `tn`, `fp`, `fn`, `accuracy`, `precision`, `recall`, and `f1`. A zero denominator gives `0.0` for that metric. Reject empty, mismatched, or non-binary inputs.

Write your solution in the next cell, then run its check cell.

In [ ]:
def classification_metrics(y_true, y_pred):
    raise NotImplementedError("Calculate confusion counts and zero-safe metrics")


In [ ]:
metrics = classification_metrics([1, 0, 1, 0, 1], [1, 0, 0, 0, 1])
check_exercise("AIML 02 · Classification metrics", [
    ("confusion counts", (metrics["tp"], metrics["tn"], metrics["fp"], metrics["fn"]) == (2, 2, 0, 1)),
    ("accuracy", metrics["accuracy"] == 0.8),
    ("precision", metrics["precision"] == 1.0),
    ("recall", abs(metrics["recall"] - 2 / 3) < 1e-12),
    ("F1", abs(metrics["f1"] - 0.8) < 1e-12),
    ("zero denominator is safe", classification_metrics([0], [0])["recall"] == 0.0),
    ("mismatched lengths rejected", raises(ValueError, lambda: classification_metrics([1], []))),
    ("non-binary label rejected", raises(ValueError, lambda: classification_metrics([2], [1]))),
])


## AIML 03 · Fit a standard scaler on training data

**Preprocessing and leakage prevention**

**Problem.** Implement `StandardScaler.fit(rows)` and `transform(rows)`. Fit per-column population means and standard deviations; a constant column uses scale 1. Transform must use the fitted training statistics, not statistics from the rows being transformed. Reject empty/ragged fit data and transform calls before fit.

Write your solution in the next cell, then run its check cell.

In [ ]:
class StandardScaler:
    def __init__(self):
        self.means = None
        self.scales = None

    def fit(self, rows):
        raise NotImplementedError("Learn per-column statistics from training rows")

    def transform(self, rows):
        raise NotImplementedError("Apply the learned statistics")


In [ ]:
scaler = StandardScaler().fit([[1, 5], [3, 5], [5, 5]])
scaled = scaler.transform([[3, 5], [7, 5]])
check_exercise("AIML 03 · StandardScaler", [
    ("training mean maps to zero", scaled[0] == [0.0, 0.0]),
    ("test row uses training statistics", math.isclose(scaled[1][0], 4 / math.sqrt(8 / 3)) and scaled[1][1] == 0.0),
    ("empty fit rejected", raises(ValueError, lambda: StandardScaler().fit([]))),
    ("ragged rows rejected", raises(ValueError, lambda: StandardScaler().fit([[1], [1, 2]]))),
    ("transform before fit rejected", raises(ValueError, lambda: StandardScaler().transform([[1]]))),
])


## AIML 04 · Linear regression with gradient descent

**Model training and loss minimization**

**Problem.** Implement `fit_linear_regression(x, y, learning_rate, epochs)` for one feature and a bias. Minimize mean squared error using batch gradient descent. Return `(slope, intercept)`; reject empty/mismatched data, non-positive learning rates, and non-positive epoch counts.

Write your solution in the next cell, then run its check cell.

In [ ]:
def fit_linear_regression(x, y, learning_rate=0.05, epochs=500):
    raise NotImplementedError("Fit slope and intercept with batch gradient descent")


In [ ]:
slope, intercept = fit_linear_regression([1, 2, 3], [3, 5, 7])
check_exercise("AIML 04 · Linear regression", [
    ("learns the slope", abs(slope - 2) < 0.01),
    ("learns the intercept", abs(intercept - 1) < 0.02),
    ("mismatched data rejected", raises(ValueError, lambda: fit_linear_regression([1], [1, 2]))),
    ("invalid learning rate rejected", raises(ValueError, lambda: fit_linear_regression([1], [1], 0))),
    ("invalid epoch count rejected", raises(ValueError, lambda: fit_linear_regression([1], [1], epochs=0))),
])


## AIML 05 · K-nearest-neighbors classification

**Distance-based modeling**

**Problem.** Implement `knn_predict(train_x, train_y, query, k=3)` with Euclidean distance. Return the most frequent label among the nearest `k` rows; break vote ties by choosing the smallest label. Reject empty/mismatched training data, invalid `k`, and feature-length mismatches.

Write your solution in the next cell, then run its check cell.

In [ ]:
def knn_predict(train_x, train_y, query, k=3):
    raise NotImplementedError("Find nearest rows and vote deterministically")


In [ ]:
features = [[0, 0], [0, 2], [3, 3], [4, 3]]
labels = [0, 1, 1, 0]
check_exercise("AIML 05 · K-nearest neighbors", [
    ("nearest neighbor prediction", knn_predict(features, labels, [0, 0.1], k=1) == 0),
    ("vote tie chooses smallest label", knn_predict(features, labels, [0, 1], k=2) == 0),
    ("k larger than dataset rejected", raises(ValueError, lambda: knn_predict(features, labels, [0, 0], 5))),
    ("feature mismatch rejected", raises(ValueError, lambda: knn_predict(features, labels, [0], 1))),
])


## AIML 06 · Choose a classification threshold with F1

**Imbalanced classification and decision policy**

**Problem.** Implement `best_f1_threshold(y_true, probabilities)`. Evaluate thresholds from the distinct probabilities, predicting positive when score is at least the threshold. Return `(threshold, f1)` for the highest F1. On a tie, return the higher threshold. Reject empty/mismatched inputs, non-binary labels, and scores outside `[0, 1]`.

Write your solution in the next cell, then run its check cell.

In [ ]:
def best_f1_threshold(y_true, probabilities):
    raise NotImplementedError("Search score thresholds and maximize F1")


In [ ]:
threshold, f1 = best_f1_threshold([0, 1, 1, 0], [0.2, 0.6, 0.8, 0.4])
check_exercise("AIML 06 · F1 threshold tuning", [
    ("best threshold selected", threshold == 0.6),
    ("best F1 returned", abs(f1 - 1.0) < 1e-12),
    ("invalid score rejected", raises(ValueError, lambda: best_f1_threshold([1], [1.2]))),
    ("mismatched inputs rejected", raises(ValueError, lambda: best_f1_threshold([1], []))),
])


# Security, AIML, and full-stack OOP challenges

This workbook contains **50 questions** from easy to advanced: core OOP, practical AIML, defensive security engineering, and full-stack application design. Each challenge has a solution cell to write in and a separate runnable checker. The complete answer key is at the very end. Security exercises focus on protective coding and safe application design.

## 24 · Password hashing and verification

**Security · Intermediate**

**Problem.** Implement `PasswordHasher`. Use PBKDF2-HMAC-SHA256 with a fresh random salt and 600,000 iterations. Store no plaintext: `hash_password` returns `salt_hex:digest_hex`; `verify` safely compares a password to that string. Reject empty passwords and malformed hashes.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class PasswordHasher:
    iterations = 600_000

    def hash_password(self, password):
        raise NotImplementedError("Hash using a fresh salt and PBKDF2")

    def verify(self, password, encoded_hash):
        raise NotImplementedError("Parse, derive, and compare safely")


In [ ]:
hasher = PasswordHasher()
saved = hasher.hash_password("correct horse battery staple")
check_exercise("24 ? PasswordHasher", [
    ("plaintext is not stored", "correct horse battery staple" not in saved),
    ("different salts produce different hashes", saved != hasher.hash_password("correct horse battery staple")),
    ("correct password verifies", hasher.verify("correct horse battery staple", saved)),
    ("wrong password does not verify", not hasher.verify("wrong password", saved)),
    ("malformed hash rejected", raises(ValueError, lambda: hasher.verify("x", "bad"))),
    ("empty salt or digest rejected", raises(ValueError, lambda: hasher.verify("x", ":"))),
])


## 25 · Role-based access control

**Security · Beginner → intermediate**

**Problem.** Implement `AccessController(user_roles, role_permissions)`. `authorize(user, permission)` returns `True` only when one of the user's roles grants that permission; otherwise raise `PermissionError`. Unknown users deny access.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class AccessController:
    def __init__(self, user_roles, role_permissions):
        raise NotImplementedError("Store the role and permission mappings")

    def authorize(self, user, permission):
        raise NotImplementedError("Check grants and deny by default")


In [ ]:
acl = AccessController(
    {"alex": ["reader"], "sam": ["admin"]},
    {"reader": {"report:read"}, "admin": {"report:read", "user:delete"}},
)
check_exercise("25 · AccessController", [
    ("granted permission", acl.authorize("alex", "report:read")),
    ("missing permission denied", raises(PermissionError, lambda: acl.authorize("alex", "user:delete"))),
    ("unknown user denied", raises(PermissionError, lambda: acl.authorize("nobody", "report:read"))),
])


## 26 · Redacted security audit events

**Security · Intermediate**

**Problem.** Implement `AuditLog.record(event, details)` so it stores a defensive copy and redacts values whose keys are `password`, `token`, `access_token`, `secret`, or `authorization` (case-insensitive). Redact matching keys inside nested dictionaries and lists too.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class AuditLog:
    def __init__(self):
        self.events = []

    def record(self, event, details):
        raise NotImplementedError("Redact sensitive fields before storing")


In [ ]:
audit = AuditLog()
details = {"user": "alex", "access_token": "do-not-log", "meta": [{"Password": "hidden"}]}
entry = audit.record("login", details)
check_exercise("26 · AuditLog", [
    ("event name recorded", entry["event"] == "login"),
    ("token redacted", entry["details"]["access_token"] == "[REDACTED]"),
    ("nested secret redacted", entry["details"]["meta"][0]["Password"] == "[REDACTED]"),
    ("input is not modified", details["access_token"] == "do-not-log"),
])


## 27 · Allowlisted upload validation

**Security · Beginner**

**Problem.** Implement `validate_upload(filename, size, allowed_extensions, max_bytes)`. Return the lowercase filename on success. Reject path separators, `.`/`..`, disallowed extensions, negative sizes, and files over the size limit with `ValueError`. This validates metadata only; do not write or execute uploaded content.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
def validate_upload(filename, size, allowed_extensions, max_bytes):
    raise NotImplementedError("Validate a plain filename, extension, and size")


In [ ]:
check_exercise("27 · Upload validation", [
    ("allowed extension is normalized", validate_upload("Report.CSV", 10, {"csv"}, 20) == "report.csv"),
    ("path separator rejected", raises(ValueError, lambda: validate_upload("../report.csv", 10, {"csv"}, 20))),
    ("unapproved extension rejected", raises(ValueError, lambda: validate_upload("script.py", 10, {"csv"}, 20))),
    ("oversized file rejected", raises(ValueError, lambda: validate_upload("a.csv", 21, {"csv"}, 20))),
])


## 28 · Confine file paths to a storage directory

**Security · Intermediate**

**Problem.** Implement `storage_path(root, filename)` for a server-controlled storage root. Accept a single plain filename and return its resolved path beneath `root`. Reject absolute paths, separators, and names that escape the storage root.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
from pathlib import Path

def storage_path(root, filename):
    raise NotImplementedError("Resolve a plain filename inside the trusted root")


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    root = Path(directory)
    safe = storage_path(root, "image.png")
    check_exercise("28 · Storage path confinement", [
        ("safe name stays in root", safe.parent == root.resolve()),
        ("parent traversal rejected", raises(ValueError, lambda: storage_path(root, "../outside.txt"))),
        ("absolute path rejected", raises(ValueError, lambda: storage_path(root, str(root.parent / "outside.txt")))),
        ("nested path rejected", raises(ValueError, lambda: storage_path(root, "sub/file.txt"))),
    ])


## 29 · Login lockout guard

**Security · Intermediate**

**Problem.** Implement `LoginGuard(max_attempts, lockout_seconds, clock)`. Track failed attempts by account. `can_attempt` is false during lockout; a success clears the account state. Inject the clock for deterministic tests.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class LoginGuard:
    def __init__(self, max_attempts, lockout_seconds, clock):
        raise NotImplementedError("Store policy, clock, and per-account state")

    def can_attempt(self, account):
        raise NotImplementedError("Check whether an account is currently locked")

    def failed(self, account):
        raise NotImplementedError("Record a failure and start lockout at the limit")

    def succeeded(self, account):
        raise NotImplementedError("Clear failures after successful authentication")


In [ ]:
class FakeClock:
    now = 100.0
    def __call__(self):
        return self.now

clock = FakeClock()
guard = LoginGuard(2, 30, clock)
guard.failed("alex")
guard.failed("alex")
check_exercise("29 · LoginGuard", [
    ("locks at configured attempts", not guard.can_attempt("alex")),
    ("other accounts remain independent", guard.can_attempt("sam")),
    ("lock expires", (setattr(clock, "now", 131.0) is None) and guard.can_attempt("alex")),
    ("success clears failures", (guard.failed("alex") is None) and (guard.succeeded("alex") is None) and guard.can_attempt("alex")),
])


## 30 · Opaque, expiring sessions

**Security · Advanced**

**Problem.** Implement `SessionStore(ttl_seconds, clock)`. `create(user_id)` returns an unguessable token. Store only a SHA-256 digest of the token, validate expiry using the injected clock, and support revocation. Never expose stored digests.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class SessionStore:
    def __init__(self, ttl_seconds, clock):
        raise NotImplementedError("Initialize an expiring session store")

    def create(self, user_id):
        raise NotImplementedError("Generate a token and store only its digest")

    def user_for(self, token):
        raise NotImplementedError("Return an active session's user or None")

    def revoke(self, token):
        raise NotImplementedError("Revoke an active token")


In [ ]:
class SessionClock:
    now = 10.0
    def __call__(self):
        return self.now

clock = SessionClock()
sessions = SessionStore(5, clock)
token = sessions.create("user-8")
def session_expires():
    expiring_token = sessions.create("user-9")
    clock.now = 16.0
    return sessions.user_for(expiring_token) is None
check_exercise("30 ? SessionStore", [
    ("token resolves to user", sessions.user_for(token) == "user-8"),
    ("raw token is not stored", token not in repr(sessions._sessions)),
    ("revocation removes access", sessions.revoke(token) and sessions.user_for(token) is None),
    ("expired token is rejected", session_expires()),
])


## 31 · Fail-closed secret configuration

**Security · Beginner → intermediate**

**Problem.** Implement `AppSettings(secret_key)`. Do not invent a default secret. Reject missing or shorter-than-32-character secrets. Its `repr` must redact the secret.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class AppSettings:
    def __init__(self, secret_key):
        raise NotImplementedError("Require a sufficiently long secret")

    def __repr__(self):
        raise NotImplementedError("Do not reveal the secret")


In [ ]:
settings = AppSettings("s" * 32)
check_exercise("31 · AppSettings", [
    ("valid secret accepted", settings.secret_key == "s" * 32),
    ("secret redacted from repr", "s" * 32 not in repr(settings)),
    ("missing secret rejected", raises(ValueError, lambda: AppSettings(None))),
    ("short secret rejected", raises(ValueError, lambda: AppSettings("short"))),
])


## 32 · Parameterized database access

**Security · Intermediate · Full-stack**

**Problem.** Implement `UserRepository.find_by_email(email)` using a DB-API-style `execute(statement, parameters)` method. Keep user input in the parameter tuple; do not interpolate it into SQL. Return the first row or `None`.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class UserRepository:
    def __init__(self, database):
        self.database = database

    def find_by_email(self, email):
        raise NotImplementedError("Use a parameterized query")


In [ ]:
class RecordingDatabase:
    def execute(self, statement, parameters):
        self.statement = statement
        self.parameters = parameters
        return [("u-1", "a@example.com")]

repository = UserRepository(RecordingDatabase())
row = repository.find_by_email("a@example.com' OR 'x'='x")
check_exercise("32 · Parameterized repository", [
    ("user value passed separately", repository.database.parameters == ("a@example.com' OR 'x'='x",)),
    ("query uses a placeholder", "?" in repository.database.statement),
    ("row returned", row == ("u-1", "a@example.com")),
])


## 33 · Per-user request rate limiter

**Security · Advanced**

**Problem.** Implement a fixed-window `RateLimiter(limit, window_seconds, clock)`. `allow(key)` returns true for at most `limit` calls per window, then false. Windows reset after their duration. Inject a clock for tests.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class RateLimiter:
    def __init__(self, limit, window_seconds, clock):
        raise NotImplementedError("Initialize per-key window state")

    def allow(self, key):
        raise NotImplementedError("Allow only the configured requests in each window")


In [ ]:
class RateClock:
    now = 0.0
    def __call__(self):
        return self.now

clock = RateClock()
limiter = RateLimiter(2, 10, clock)
check_exercise("33 · RateLimiter", [
    ("first two requests allowed", limiter.allow("ip-a") and limiter.allow("ip-a")),
    ("limit enforced", not limiter.allow("ip-a")),
    ("other key is independent", limiter.allow("ip-b")),
    ("new window resets", (setattr(clock, "now", 11.0) is None) and limiter.allow("ip-a")),
])


## 34 · Regression evaluation metrics

**AIML · Beginner**

**Problem.** Implement `regression_metrics(actual, predicted)` returning mean absolute error, mean squared error, and root mean squared error. Reject empty or mismatched inputs.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
def regression_metrics(actual, predicted):
    raise NotImplementedError("Calculate MAE, MSE, and RMSE")


In [ ]:
metrics = regression_metrics([3, -0.5, 2, 7], [2.5, 0, 2, 8])
check_exercise("34 · Regression metrics", [
    ("MAE", math.isclose(metrics["mae"], 0.5)),
    ("MSE", math.isclose(metrics["mse"], 0.375)),
    ("RMSE", math.isclose(metrics["rmse"], math.sqrt(0.375))),
    ("empty inputs rejected", raises(ValueError, lambda: regression_metrics([], []))),
    ("mismatched inputs rejected", raises(ValueError, lambda: regression_metrics([1], []))),
])


## 35 · Deterministic K-fold cross-validation

**AIML · Intermediate**

**Problem.** Implement `kfold_indices(row_count, k)`. Return `k` `(train, validation)` pairs. Every row appears in validation exactly once; folds differ in size by at most one. Reject `k < 2` or `k > row_count`.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
def kfold_indices(row_count, k):
    raise NotImplementedError("Partition rows into complete validation folds")


In [ ]:
folds = kfold_indices(11, 4)
validation_rows = [row for _, validation in folds for row in validation]
check_exercise("35 · K-fold splitting", [
    ("requested fold count", len(folds) == 4),
    ("every row validates once", sorted(validation_rows) == list(range(11))),
    ("train and validation are disjoint", all(set(train).isdisjoint(valid) for train, valid in folds)),
    ("fold sizes balanced", max(map(len, [valid for _, valid in folds])) - min(map(len, [valid for _, valid in folds])) <= 1),
    ("invalid k rejected", raises(ValueError, lambda: kfold_indices(3, 4))),
])


## 36 · Early stopping for model training

**AIML · Intermediate**

**Problem.** Implement `EarlyStopping(patience, min_delta=0)`. `update(metric)` returns true when training should stop. A lower metric is better. Only an improvement greater than `min_delta` resets patience.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class EarlyStopping:
    def __init__(self, patience, min_delta=0):
        raise NotImplementedError("Initialize best score and patience state")

    def update(self, metric):
        raise NotImplementedError("Track improvement and signal when to stop")


In [ ]:
stopper = EarlyStopping(patience=2, min_delta=0.1)
check_exercise("36 · EarlyStopping", [
    ("first score continues", not stopper.update(1.0)),
    ("meaningful improvement continues", not stopper.update(0.8)),
    ("small change counts as no improvement", not stopper.update(0.75)),
    ("patience exhaustion stops", stopper.update(0.74)),
])


## 37 · Soft-voting classifier ensemble

**AIML · Intermediate**

**Problem.** Implement `SoftVotingEnsemble(models)`. Each model's `predict_proba(row)` returns class probabilities in the same class order. Average probabilities and predict the index of the largest average. Reject an empty ensemble.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class SoftVotingEnsemble:
    def __init__(self, models):
        raise NotImplementedError("Store a non-empty collection of models")

    def predict_proba(self, row):
        raise NotImplementedError("Average model probability vectors")

    def predict(self, row):
        raise NotImplementedError("Return the class with highest average probability")


In [ ]:
class FixedModel:
    def __init__(self, probabilities):
        self.probabilities = probabilities
    def predict_proba(self, row):
        return self.probabilities

ensemble = SoftVotingEnsemble([FixedModel([0.8, 0.2]), FixedModel([0.3, 0.7])])
check_exercise("37 · Soft voting", [
    ("probabilities averaged", ensemble.predict_proba([1]) == [0.55, 0.44999999999999996]),
    ("highest average class selected", ensemble.predict([1]) == 0),
    ("empty ensemble rejected", raises(ValueError, lambda: SoftVotingEnsemble([]))),
])


## 38 · Feature drift detector

**AIML · Intermediate → advanced**

**Problem.** Implement `MeanDriftDetector(reference_rows, threshold)`. `detect(rows)` returns a boolean per feature, true when its mean differs from the reference mean by more than `threshold`. Reject empty or mismatched feature dimensions.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class MeanDriftDetector:
    def __init__(self, reference_rows, threshold):
        raise NotImplementedError("Store reference feature means")

    def detect(self, rows):
        raise NotImplementedError("Compare incoming feature means")


In [ ]:
detector = MeanDriftDetector([[1, 10], [3, 12]], threshold=2)
check_exercise("38 · Mean drift detector", [
    ("stable means do not drift", detector.detect([[2, 11], [2, 11]]) == [False, False]),
    ("shifted feature is detected", detector.detect([[8, 11], [8, 11]]) == [True, False]),
    ("empty batch rejected", raises(ValueError, lambda: detector.detect([]))),
])


## 39 · Model artifact integrity

**AIML · Security · Intermediate**

**Problem.** Implement `ModelArtifact(data)`. Calculate and retain a SHA-256 checksum. `verify(candidate)` uses a constant-time comparison and returns false if bytes have changed. Do not deserialize or execute the artifact.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class ModelArtifact:
    def __init__(self, data):
        raise NotImplementedError("Store bytes and their SHA-256 checksum")

    def verify(self, candidate):
        raise NotImplementedError("Compare candidate checksum safely")


In [ ]:
artifact = ModelArtifact(b"trained-model-bytes")
check_exercise("39 · Model artifact integrity", [
    ("original bytes verify", artifact.verify(b"trained-model-bytes")),
    ("modified bytes fail verification", not artifact.verify(b"modified-model-bytes")),
    ("checksum is a SHA-256 hex digest", len(artifact.checksum) == 64),
])


## 40 · Versioned feature store

**AIML · Advanced**

**Problem.** Implement `FeatureStore.put(entity, version, values)` and `get(entity, version)`. Return defensive copies so callers cannot mutate stored features. Raise `KeyError` for an unknown entity/version pair.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class FeatureStore:
    def __init__(self):
        raise NotImplementedError("Initialize versioned feature storage")

    def put(self, entity, version, values):
        raise NotImplementedError("Store a defensive copy")

    def get(self, entity, version):
        raise NotImplementedError("Return a defensive copy or raise KeyError")


In [ ]:
store = FeatureStore()
store.put("customer-1", "v2", {"age": 31, "score": [0.7]})
features = store.get("customer-1", "v2")
features["score"].append(1.0)
check_exercise("40 · FeatureStore", [
    ("nested values copied on read", store.get("customer-1", "v2")["score"] == [0.7]),
    ("unknown version rejected", raises(KeyError, lambda: store.get("customer-1", "v9"))),
])


## 41 · Promote a model only when its metric improves

**AIML · Advanced**

**Problem.** Implement `ModelCatalog(initial_name, initial_score)`. `consider(name, score)` promotes a candidate only if its score is strictly higher. Return whether promotion occurred and expose the current best model.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class ModelCatalog:
    def __init__(self, initial_name, initial_score):
        raise NotImplementedError("Initialize the current champion")

    def consider(self, name, score):
        raise NotImplementedError("Promote only a strictly better candidate")


In [ ]:
catalog = ModelCatalog("v1", 0.82)
check_exercise("41 · Model promotion", [
    ("better model promoted", catalog.consider("v2", 0.9) and catalog.champion == "v2"),
    ("tie does not replace champion", not catalog.consider("v3", 0.9) and catalog.champion == "v2"),
    ("worse model does not replace champion", not catalog.consider("v4", 0.7)),
])


## 42 · Bounded least-recently-used inference cache

**AIML · Advanced**

**Problem.** Implement `LRUCache(capacity)` with `get(key)` and `put(key, value)`. A get refreshes recency; inserting beyond capacity evicts the least recently used entry. Missing keys return `None`.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        raise NotImplementedError("Initialize a bounded cache")

    def get(self, key):
        raise NotImplementedError("Read and refresh recency")

    def put(self, key, value):
        raise NotImplementedError("Insert and evict least-recently-used entry")


In [ ]:
cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")
cache.put("c", 3)
check_exercise("42 · LRUCache", [
    ("recently used entry retained", cache.get("a") == 1),
    ("least recently used entry evicted", cache.get("b") is None),
    ("new value retained", cache.get("c") == 3),
    ("invalid capacity rejected", raises(ValueError, lambda: LRUCache(0))),
])


## 43 · Discount strategies for a shopping cart

**Full-stack · Beginner**

**Problem.** Implement `ShoppingCart(discount_strategy)`. `total(items)` sums item prices and delegates the discount calculation to the injected strategy. Include `NoDiscount` and `PercentDiscount(rate)`; reject rates outside `[0, 1]`.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class NoDiscount:
    def apply(self, subtotal):
        raise NotImplementedError("Return the unchanged subtotal")

class PercentDiscount:
    def __init__(self, rate):
        raise NotImplementedError("Validate and store the discount rate")

    def apply(self, subtotal):
        raise NotImplementedError("Apply the percentage discount")

class ShoppingCart:
    def __init__(self, discount_strategy):
        raise NotImplementedError("Inject the discount strategy")

    def total(self, items):
        raise NotImplementedError("Calculate subtotal and apply strategy")


In [ ]:
cart = ShoppingCart(PercentDiscount(0.1))
check_exercise("43 · ShoppingCart strategies", [
    ("discount applied", math.isclose(cart.total([10, 20]), 27)),
    ("no discount strategy", ShoppingCart(NoDiscount()).total([10, 20]) == 30),
    ("invalid rate rejected", raises(ValueError, lambda: PercentDiscount(1.5))),
])


## 44 · Validated API pagination

**Full-stack · Beginner → intermediate**

**Problem.** Implement `paginate(items, page, page_size, max_page_size=100)`. Pages are 1-based. Return a dictionary with `items`, `page`, `page_size`, and `total`. Reject invalid page values or page sizes.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
def paginate(items, page, page_size, max_page_size=100):
    raise NotImplementedError("Validate and return one page of results")


In [ ]:
result = paginate(list(range(23)), page=3, page_size=10)
check_exercise("44 · Pagination", [
    ("correct slice", result["items"] == [20, 21, 22]),
    ("metadata included", (result["page"], result["page_size"], result["total"]) == (3, 10, 23)),
    ("page zero rejected", raises(ValueError, lambda: paginate([], 0, 10))),
    ("oversized page rejected", raises(ValueError, lambda: paginate([], 1, 101))),
])


## 45 · Publish/subscribe event bus

**Full-stack · Intermediate · Observer pattern**

**Problem.** Implement `EventBus.subscribe(event, handler)` and `publish(event, payload)`. Call subscribers in registration order. Return an unsubscribe function; publishing an event without subscribers is a no-op.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class EventBus:
    def __init__(self):
        raise NotImplementedError("Initialize event subscriptions")

    def subscribe(self, event, handler):
        raise NotImplementedError("Register a handler and return unsubscribe")

    def publish(self, event, payload):
        raise NotImplementedError("Notify handlers in order")


In [ ]:
bus = EventBus()
received = []
unsubscribe = bus.subscribe("created", lambda value: received.append(("first", value)))
bus.subscribe("created", lambda value: received.append(("second", value)))
bus.publish("created", 7)
unsubscribe()
bus.publish("created", 8)
check_exercise("45 · EventBus", [
    ("subscribers called in order", received[:2] == [("first", 7), ("second", 7)]),
    ("unsubscribed handler not called", received[-1] == ("second", 8) and len(received) == 3),
])


## 46 · Expiring in-memory cache

**Full-stack · Intermediate**

**Problem.** Implement `TTLCache(clock)`. `set(key, value, ttl)` stores a value until `clock() + ttl`; `get(key)` returns `None` after expiry and removes the entry. Reject non-positive TTLs.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class TTLCache:
    def __init__(self, clock):
        raise NotImplementedError("Store clock and cache entries")

    def set(self, key, value, ttl):
        raise NotImplementedError("Store value and expiration")

    def get(self, key):
        raise NotImplementedError("Return a live value or None")


In [ ]:
class CacheClock:
    now = 4.0
    def __call__(self):
        return self.now

clock = CacheClock()
cache = TTLCache(clock)
cache.set("profile", {"name": "A"}, 3)
check_exercise("46 · TTLCache", [
    ("live value returned", cache.get("profile") == {"name": "A"}),
    ("expired value removed", (setattr(clock, "now", 7.0) is None) and cache.get("profile") is None),
    ("invalid TTL rejected", raises(ValueError, lambda: cache.set("x", 1, 0))),
])


## 47 · Bounded retry policy

**Full-stack · Intermediate**

**Problem.** Implement `RetryPolicy(max_attempts, retry_on).run(operation)`. Retry only the specified exception types and stop after the configured total attempts. Return the operation result or re-raise the final/non-retryable error.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class RetryPolicy:
    def __init__(self, max_attempts, retry_on):
        raise NotImplementedError("Validate and store retry configuration")

    def run(self, operation):
        raise NotImplementedError("Retry only allowed errors, with a bounded attempt count")


In [ ]:
class TemporaryFailure(Exception):
    pass

calls = []
def flaky():
    calls.append(1)
    if len(calls) < 3:
        raise TemporaryFailure()
    return "ok"

policy = RetryPolicy(3, (TemporaryFailure,))
check_exercise("47 · RetryPolicy", [
    ("eventual success returned", policy.run(flaky) == "ok"),
    ("bounded retry count", len(calls) == 3),
    ("non-retryable error propagates", raises(ValueError, lambda: policy.run(lambda: (_ for _ in ()).throw(ValueError())))),
    ("invalid attempt count rejected", raises(ValueError, lambda: RetryPolicy(0, (Exception,)))),
])


## 48 · Unit of work with rollback

**Full-stack · Advanced · Transaction boundaries**

**Problem.** Implement `UnitOfWork(database)` as a context manager. Begin on entry; commit on success; roll back if the block raises; never swallow the original exception.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class UnitOfWork:
    def __init__(self, database):
        self.database = database

    def __enter__(self):
        raise NotImplementedError("Begin and return the unit of work")

    def __exit__(self, exception_type, exception, traceback):
        raise NotImplementedError("Commit or roll back and propagate failures")


In [ ]:
class FakeDatabase:
    def __init__(self):
        self.events = []
    def begin(self):
        self.events.append("begin")
    def commit(self):
        self.events.append("commit")
    def rollback(self):
        self.events.append("rollback")

database = FakeDatabase()
with UnitOfWork(database):
    pass
def failed_transaction():
    with UnitOfWork(database):
        raise RuntimeError("fail")
check_exercise("48 · UnitOfWork", [
    ("successful transaction commits", database.events[:2] == ["begin", "commit"]),
    ("exception propagates", raises(RuntimeError, failed_transaction)),
    ("failed transaction rolls back", database.events[-2:] == ["begin", "rollback"]),
])


## 49 · Validated create-user request object

**Full-stack · Intermediate · API boundary**

**Problem.** Implement `CreateUserRequest.from_payload(payload)`. Require exactly the keys `email` and `name`, both non-empty strings. Trim both values and lowercase the email. Invalid input raises `ValueError`.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class CreateUserRequest:
    def __init__(self, email, name):
        self.email = email
        self.name = name

    @classmethod
    def from_payload(cls, payload):
        raise NotImplementedError("Validate and normalize the request")


In [ ]:
request = CreateUserRequest.from_payload({"email": " A@EXAMPLE.COM ", "name": " Ada "})
check_exercise("49 · CreateUserRequest", [
    ("fields normalized", (request.email, request.name) == ("a@example.com", "Ada")),
    ("missing key rejected", raises(ValueError, lambda: CreateUserRequest.from_payload({"email": "a@b.c"}))),
    ("unexpected key rejected", raises(ValueError, lambda: CreateUserRequest.from_payload({"email": "a@b.c", "name": "A", "admin": True}))),
    ("non-string rejected", raises(ValueError, lambda: CreateUserRequest.from_payload({"email": "a@b.c", "name": 2}))),
])


## 50 · Order service with injected payment and repository

**Full-stack · Advanced · Dependency inversion**

**Problem.** Implement `OrderService(repository, payment_gateway).place_order(user_id, items)`. Reject empty carts or non-positive item prices. Charge the total through the injected gateway; save and return an order only after payment succeeds. A declined payment raises `ValueError` and is not saved.

Implement the behavior in the next cell, then run its check cell.

In [ ]:
class OrderService:
    def __init__(self, repository, payment_gateway):
        raise NotImplementedError("Store injected collaborators")

    def place_order(self, user_id, items):
        raise NotImplementedError("Validate, charge, then persist an order")


In [ ]:
class FakePayments:
    def __init__(self, approved=True):
        self.approved = approved
        self.charges = []
    def charge(self, user_id, amount):
        self.charges.append((user_id, amount))
        return self.approved

class FakeOrders:
    def __init__(self):
        self.saved = []
    def save(self, order):
        self.saved.append(order)
        return order

payments = FakePayments()
orders = FakeOrders()
service = OrderService(orders, payments)
order = service.place_order("u1", [10, 15])
check_exercise("50 · OrderService", [
    ("gateway charged correct total", payments.charges == [("u1", 25)]),
    ("successful order persisted", order["total"] == 25 and orders.saved == [order]),
    ("empty cart rejected", raises(ValueError, lambda: service.place_order("u1", []))),
    ("declined payment not persisted", raises(ValueError, lambda: OrderService(orders, FakePayments(False)).place_order("u2", [5])) and len(orders.saved) == 1),
])


# Answer key — open after attempting the exercises

Solutions are collected here so the challenges and checks remain usable without revealing answers immediately. Compare the design and edge-case handling with your implementation; copying is optional.

### 01 · Classes, objects, attributes, and methods

```python
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def label(self):
        return f"{self.title} ({self.pages} pages)"
```

### 02 · Encapsulation, properties, and validated state

```python
class BankAccount:
    def __init__(self, owner, opening_balance=0):
        if opening_balance < 0:
            raise ValueError("opening balance cannot be negative")
        self.owner = owner
        self._balance = opening_balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        return self._balance

    def withdraw(self, amount):
        if amount <= 0 or amount > self._balance:
            raise ValueError("invalid withdrawal")
        self._balance -= amount
        return self._balance
```

### 03 · Class state, class methods, and static methods

```python
class TrainingRun:
    created_count = 0

    def __init__(self, model_name):
        self.model_name = model_name
        type(self).created_count += 1

    @classmethod
    def count(cls):
        return cls.created_count

    @staticmethod
    def valid_epochs(value):
        return isinstance(value, int) and not isinstance(value, bool) and value > 0
```

### 04 · Dunder methods and value objects

```python
class PredictionResult:
    def __init__(self, label, confidence):
        if not 0 <= confidence <= 1:
            raise ValueError("confidence must be between 0 and 1")
        self.label = label
        self.confidence = confidence

    def __str__(self):
        return f"{self.label}: {self.confidence:.0%}"

    def __eq__(self, other):
        if not isinstance(other, PredictionResult):
            return NotImplemented
        return (self.label, self.confidence) == (other.label, other.confidence)
```

### 05 · Inheritance, overriding, and super()

```python
class Predictor:
    def __init__(self, name):
        self.name = name

    def predict(self, features):
        raise NotImplementedError("subclasses must implement predict")


class LinearPredictor(Predictor):
    def __init__(self, weights, bias=0):
        super().__init__("linear")
        self.weights = list(weights)
        self.bias = bias

    def predict(self, features):
        if len(features) != len(self.weights):
            raise ValueError("feature count must match weight count")
        return sum(w * x for w, x in zip(self.weights, features)) + self.bias
```

### 06 · Polymorphism and duck typing

```python
def batch_predict(models, features):
    return [model.predict(features) for model in models]
```

### 07 · Composition and preprocessing pipelines

```python
class FeaturePipeline:
    def __init__(self, transformers):
        self.transformers = list(transformers)

    def transform(self, values):
        result = list(values)
        for transformer in self.transformers:
            result = transformer.transform(result)
        return result
```

### 08 · Abstract base classes and contracts

```python
from abc import ABC, abstractmethod

class Storage(ABC):
    @abstractmethod
    def save(self, key, value):
        raise NotImplementedError

    @abstractmethod
    def load(self, key):
        raise NotImplementedError


class InMemoryStorage(Storage):
    def __init__(self):
        self._values = {}

    def save(self, key, value):
        self._values[key] = value

    def load(self, key):
        return self._values.get(key)
```

### 09 · Dataclasses and custom validation errors

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class FeatureVector:
    names: tuple
    values: tuple

    def __post_init__(self):
        if len(self.names) != len(self.values):
            raise ValueError("each feature name needs one value")


class DataValidationError(ValueError):
    pass


class DatasetBatch:
    def __init__(self, features, labels):
        if not features or len(features) != len(labels) or any(not row for row in features):
            raise DataValidationError("features and labels need matching non-empty rows")
        self.features = [list(row) for row in features]
        self.labels = list(labels)

    def __len__(self):
        return len(self.features)
```

### 10 · Mixins and method resolution order

```python
import json

class TimestampMixin:
    def timestamp(self):
        return self.created_at

class JSONExportMixin:
    def to_json(self):
        return json.dumps(self.__dict__, sort_keys=True)

class TrainingCard(TimestampMixin, JSONExportMixin):
    def __init__(self, model_name, created_at):
        self.model_name = model_name
        self.created_at = created_at
```

### 11 · Descriptors and reusable attribute validation

```python
class PositiveFloat:
    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
            raise ValueError("value must be a positive number")
        setattr(instance, self.storage_name, float(value))


class TrainingConfig:
    learning_rate = PositiveFloat()

    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
```

### 12 · Protocols and structural typing

```python
from typing import Protocol

class PredictorProtocol(Protocol):
    def predict(self, features: list[float]) -> float:
        ...

def run_prediction(model: PredictorProtocol, features: list[float]) -> float:
    return model.predict(features)
```

### 13 · Repository and service layers

```python
class UserRepository:
    def __init__(self):
        self._users = {}

    def find_by_email(self, email):
        return self._users.get(email)

    def add(self, user):
        self._users[user["email"]] = user


class UserService:
    def __init__(self, repository):
        self.repository = repository

    def register(self, email, name):
        email = email.strip().lower()
        name = name.strip()
        if not email or not name:
            raise ValueError("email and name are required")
        if self.repository.find_by_email(email) is not None:
            raise ValueError("email is already registered")
        user = {"email": email, "name": name}
        self.repository.add(user)
        return user
```

### 14 · Controller layer and API response shape

```python
class UserController:
    def __init__(self, user_service):
        self.user_service = user_service

    def create(self, payload):
        try:
            user = self.user_service.register(payload["email"], payload["name"])
        except ValueError as error:
            return {"status": 400, "body": {"error": str(error)}}
        return {"status": 201, "body": user}
```

### 15 · AIML serving, composition, and dependency injection

```python
class ServingService:
    def __init__(self, preprocessor, model, threshold=0.5):
        if not 0 <= threshold <= 1:
            raise ValueError("threshold must be between 0 and 1")
        self.preprocessor = preprocessor
        self.model = model
        self.threshold = threshold

    def predict(self, raw_features):
        features = self.preprocessor.transform(raw_features)
        probabilities = self.model.predict_proba(features)
        positive_probability = probabilities[1]
        return {
            "label": int(positive_probability >= self.threshold),
            "probability": positive_probability,
        }
```

### 16 · Class-level registries and factories

```python
class ModelRegistry:
    _factories = {}

    @classmethod
    def register(cls, name, factory):
        if not name or not callable(factory):
            raise ValueError("a name and callable factory are required")
        cls._factories[name] = factory

    @classmethod
    def create(cls, name):
        try:
            factory = cls._factories[name]
        except KeyError:
            raise KeyError(f"unknown model: {name}") from None
        return factory()
```

### 17 · Context managers and transactional cleanup

```python
class AtomicList:
    def __init__(self, values):
        self.values = list(values)

    def __enter__(self):
        self._snapshot = list(self.values)
        return self

    def __exit__(self, exception_type, exception, traceback):
        if exception_type is not None:
            self.values[:] = self._snapshot
        return False
```

### AIML 01 · Train/test split without leakage

```python
def split_indices(row_count, test_ratio=0.25, seed=0):
    if row_count < 2:
        raise ValueError("at least two rows are required")
    if not 0 < test_ratio < 1:
        raise ValueError("test_ratio must be between 0 and 1")
    indices = list(range(row_count))
    random.Random(seed).shuffle(indices)
    test_count = max(1, min(row_count - 1, round(row_count * test_ratio)))
    return indices[test_count:], indices[:test_count]
```

### AIML 02 · Confusion matrix and classification metrics

```python
def classification_metrics(y_true, y_pred):
    if not y_true or len(y_true) != len(y_pred):
        raise ValueError("labels must be non-empty and have equal lengths")
    if any(label not in (0, 1) for label in y_true + y_pred):
        raise ValueError("labels must be binary integers 0 or 1")

    tp = sum(actual == 1 and predicted == 1 for actual, predicted in zip(y_true, y_pred))
    tn = sum(actual == 0 and predicted == 0 for actual, predicted in zip(y_true, y_pred))
    fp = sum(actual == 0 and predicted == 1 for actual, predicted in zip(y_true, y_pred))
    fn = sum(actual == 1 and predicted == 0 for actual, predicted in zip(y_true, y_pred))
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
    }
```

### AIML 03 · Fit a standard scaler on training data

```python
class StandardScaler:
    def __init__(self):
        self.means = None
        self.scales = None

    def fit(self, rows):
        if not rows or not rows[0] or any(len(row) != len(rows[0]) for row in rows):
            raise ValueError("fit requires non-empty rows with equal feature counts")
        columns = list(zip(*rows))
        self.means = [sum(column) / len(column) for column in columns]
        self.scales = []
        for column, mean in zip(columns, self.means):
            variance = sum((value - mean) ** 2 for value in column) / len(column)
            standard_deviation = math.sqrt(variance)
            self.scales.append(standard_deviation or 1.0)
        return self

    def transform(self, rows):
        if self.means is None or self.scales is None:
            raise ValueError("fit the scaler before transforming")
        if any(len(row) != len(self.means) for row in rows):
            raise ValueError("feature count does not match fitted data")
        return [
            [(value - mean) / scale for value, mean, scale in zip(row, self.means, self.scales)]
            for row in rows
        ]
```

### AIML 04 · Linear regression with gradient descent

```python
def fit_linear_regression(x, y, learning_rate=0.05, epochs=500):
    if not x or len(x) != len(y):
        raise ValueError("x and y must be non-empty and have equal lengths")
    if learning_rate <= 0 or epochs <= 0:
        raise ValueError("learning_rate and epochs must be positive")

    slope = 0.0
    intercept = 0.0
    count = len(x)
    for _ in range(epochs):
        slope_gradient = sum(
            2 * feature * (slope * feature + intercept - target)
            for feature, target in zip(x, y)
        ) / count
        intercept_gradient = sum(
            2 * (slope * feature + intercept - target)
            for feature, target in zip(x, y)
        ) / count
        slope -= learning_rate * slope_gradient
        intercept -= learning_rate * intercept_gradient
    return slope, intercept
```

### AIML 05 · K-nearest-neighbors classification

```python
def knn_predict(train_x, train_y, query, k=3):
    if not train_x or len(train_x) != len(train_y):
        raise ValueError("training features and labels must be non-empty and aligned")
    if not 1 <= k <= len(train_x):
        raise ValueError("k must be between 1 and the number of training rows")
    if any(len(row) != len(query) for row in train_x):
        raise ValueError("query and training rows must have equal feature counts")

    neighbors = sorted(
        (
            (sum((a - b) ** 2 for a, b in zip(row, query)), label)
            for row, label in zip(train_x, train_y)
        ),
        key=lambda item: item[0],
    )[:k]
    votes = {}
    for _, label in neighbors:
        votes[label] = votes.get(label, 0) + 1
    return min(votes, key=lambda label: (-votes[label], label))
```

### AIML 06 · Choose a classification threshold with F1

```python
def best_f1_threshold(y_true, probabilities):
    if not y_true or len(y_true) != len(probabilities):
        raise ValueError("labels and probabilities must be non-empty and aligned")
    if any(label not in (0, 1) for label in y_true):
        raise ValueError("labels must be binary integers 0 or 1")
    if any(not 0 <= score <= 1 for score in probabilities):
        raise ValueError("probabilities must be between 0 and 1")

    best_threshold = None
    best_f1 = -1.0
    for threshold in sorted(set(probabilities), reverse=True):
        predictions = [int(score >= threshold) for score in probabilities]
        score = classification_metrics(y_true, predictions)["f1"]
        if score > best_f1:
            best_threshold = threshold
            best_f1 = score
    return best_threshold, best_f1
```

### 24 ? Password hashing and verification

```python
class PasswordHasher:
    iterations = 600_000

    def hash_password(self, password):
        if not isinstance(password, str) or not password:
            raise ValueError("password must be a non-empty string")
        salt = secrets.token_bytes(16)
        digest = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, self.iterations)
        return f"{salt.hex()}:{digest.hex()}"

    def verify(self, password, encoded_hash):
        if not isinstance(password, str) or not password:
            raise ValueError("password must be a non-empty string")
        try:
            salt_hex, digest_hex = encoded_hash.split(":")
            salt = bytes.fromhex(salt_hex)
            expected = bytes.fromhex(digest_hex)
        except (ValueError, AttributeError, TypeError):
            raise ValueError("malformed password hash") from None
        if len(salt) != 16 or len(expected) != 32:
            raise ValueError("malformed password hash")
        actual = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, self.iterations)
        return hmac.compare_digest(actual, expected)
```


### 25 · Role-based access control

```python
class AccessController:
    def __init__(self, user_roles, role_permissions):
        self.user_roles = user_roles
        self.role_permissions = role_permissions

    def authorize(self, user, permission):
        roles = self.user_roles.get(user, ())
        if any(permission in self.role_permissions.get(role, set()) for role in roles):
            return True
        raise PermissionError("permission denied")
```

### 26 · Redacted security audit events

```python
class AuditLog:
    _sensitive = {"password", "token", "access_token", "secret", "authorization"}

    def __init__(self):
        self.events = []

    def _redact(self, value):
        if isinstance(value, dict):
            return {
                key: "[REDACTED]" if str(key).casefold() in self._sensitive else self._redact(item)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [self._redact(item) for item in value]
        return value

    def record(self, event, details):
        entry = {"event": event, "details": self._redact(details)}
        self.events.append(entry)
        return entry
```

### 27 · Allowlisted upload validation

```python
def validate_upload(filename, size, allowed_extensions, max_bytes):
    if not isinstance(filename, str) or not filename or filename in {".", ".."}:
        raise ValueError("invalid filename")
    if "/" in filename or "\\" in filename or ".." in filename:
        raise ValueError("filename must not contain a path")
    if size < 0 or size > max_bytes:
        raise ValueError("file size is outside the allowed limit")
    normalized = filename.lower()
    extension = normalized.rsplit(".", 1)[-1] if "." in normalized else ""
    allowed = {item.lower().lstrip(".") for item in allowed_extensions}
    if extension not in allowed:
        raise ValueError("file extension is not allowed")
    return normalized
```

### 28 · Confine file paths to a storage directory

```python
from pathlib import Path

def storage_path(root, filename):
    if not isinstance(filename, str) or not filename or filename in {".", ".."}:
        raise ValueError("filename must be a plain name")
    if "/" in filename or "\\" in filename or Path(filename).is_absolute():
        raise ValueError("filename must not contain a path")
    base = Path(root).resolve()
    candidate = (base / filename).resolve()
    try:
        candidate.relative_to(base)
    except ValueError:
        raise ValueError("path escapes the storage root") from None
    return candidate
```

### 29 · Login lockout guard

```python
class LoginGuard:
    def __init__(self, max_attempts, lockout_seconds, clock):
        if max_attempts < 1 or lockout_seconds <= 0:
            raise ValueError("lockout settings must be positive")
        self.max_attempts = max_attempts
        self.lockout_seconds = lockout_seconds
        self.clock = clock
        self._failures = {}
        self._locked_until = {}

    def can_attempt(self, account):
        until = self._locked_until.get(account)
        if until is None:
            return True
        if self.clock() >= until:
            self._locked_until.pop(account, None)
            self._failures.pop(account, None)
            return True
        return False

    def failed(self, account):
        self._failures[account] = self._failures.get(account, 0) + 1
        if self._failures[account] >= self.max_attempts:
            self._locked_until[account] = self.clock() + self.lockout_seconds

    def succeeded(self, account):
        self._failures.pop(account, None)
        self._locked_until.pop(account, None)
```

### 30 · Opaque, expiring sessions

```python
class SessionStore:
    def __init__(self, ttl_seconds, clock):
        if ttl_seconds <= 0:
            raise ValueError("ttl_seconds must be positive")
        self.ttl_seconds = ttl_seconds
        self.clock = clock
        self._sessions = {}

    def _digest(self, token):
        return hashlib.sha256(token.encode("utf-8")).hexdigest()

    def create(self, user_id):
        token = secrets.token_urlsafe(32)
        self._sessions[self._digest(token)] = (user_id, self.clock() + self.ttl_seconds)
        return token

    def user_for(self, token):
        digest = self._digest(token)
        session = self._sessions.get(digest)
        if session is None:
            return None
        user_id, expires_at = session
        if self.clock() >= expires_at:
            self._sessions.pop(digest, None)
            return None
        return user_id

    def revoke(self, token):
        return self._sessions.pop(self._digest(token), None) is not None
```

### 31 · Fail-closed secret configuration

```python
class AppSettings:
    def __init__(self, secret_key):
        if not isinstance(secret_key, str) or len(secret_key) < 32:
            raise ValueError("secret_key must contain at least 32 characters")
        self.secret_key = secret_key

    def __repr__(self):
        return "AppSettings(secret_key='[REDACTED]')"
```

### 32 · Parameterized database access

```python
class UserRepository:
    def __init__(self, database):
        self.database = database

    def find_by_email(self, email):
        rows = self.database.execute(
            "SELECT id, email FROM users WHERE email = ?",
            (email,),
        )
        return rows[0] if rows else None
```

### 33 · Per-user request rate limiter

```python
class RateLimiter:
    def __init__(self, limit, window_seconds, clock):
        if limit < 1 or window_seconds <= 0:
            raise ValueError("limit and window_seconds must be positive")
        self.limit = limit
        self.window_seconds = window_seconds
        self.clock = clock
        self._windows = {}

    def allow(self, key):
        now = self.clock()
        start, count = self._windows.get(key, (now, 0))
        if now - start >= self.window_seconds:
            start, count = now, 0
        if count >= self.limit:
            return False
        self._windows[key] = (start, count + 1)
        return True
```

### 34 · Regression evaluation metrics

```python
def regression_metrics(actual, predicted):
    if not actual or len(actual) != len(predicted):
        raise ValueError("actual and predicted values must be non-empty and aligned")
    errors = [prediction - target for target, prediction in zip(actual, predicted)]
    mae = sum(abs(error) for error in errors) / len(errors)
    mse = sum(error * error for error in errors) / len(errors)
    return {"mae": mae, "mse": mse, "rmse": math.sqrt(mse)}
```

### 35 · Deterministic K-fold cross-validation

```python
def kfold_indices(row_count, k):
    if k < 2 or k > row_count:
        raise ValueError("k must be between 2 and row_count")
    indices = list(range(row_count))
    base, remainder = divmod(row_count, k)
    folds = []
    start = 0
    for fold_number in range(k):
        size = base + (fold_number < remainder)
        validation = indices[start:start + size]
        train = indices[:start] + indices[start + size:]
        folds.append((train, validation))
        start += size
    return folds
```

### 36 · Early stopping for model training

```python
class EarlyStopping:
    def __init__(self, patience, min_delta=0):
        if patience < 1 or min_delta < 0:
            raise ValueError("patience must be positive and min_delta non-negative")
        self.patience = patience
        self.min_delta = min_delta
        self.best = math.inf
        self.bad_epochs = 0

    def update(self, metric):
        if metric < self.best - self.min_delta:
            self.best = metric
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience
```

### 37 ? Soft-voting classifier ensemble

```python
class SoftVotingEnsemble:
    def __init__(self, models):
        self.models = list(models)
        if not self.models:
            raise ValueError("at least one model is required")

    def predict_proba(self, row):
        probabilities = [model.predict_proba(row) for model in self.models]
        if any(len(values) != len(probabilities[0]) for values in probabilities):
            raise ValueError("models must return equal-sized probability vectors")
        return [
            sum(values[index] for values in probabilities) / len(probabilities)
            for index in range(len(probabilities[0]))
        ]

    def predict(self, row):
        probabilities = self.predict_proba(row)
        return max(range(len(probabilities)), key=probabilities.__getitem__)
```


### 38 · Feature drift detector

```python
class MeanDriftDetector:
    def __init__(self, reference_rows, threshold):
        if not reference_rows or not reference_rows[0] or threshold < 0:
            raise ValueError("reference rows and a non-negative threshold are required")
        if any(len(row) != len(reference_rows[0]) for row in reference_rows):
            raise ValueError("reference rows must have equal feature counts")
        self.means = [
            sum(column) / len(column)
            for column in zip(*reference_rows)
        ]
        self.threshold = threshold

    def detect(self, rows):
        if not rows or any(len(row) != len(self.means) for row in rows):
            raise ValueError("incoming rows must match the fitted feature count")
        current = [sum(column) / len(column) for column in zip(*rows)]
        return [
            abs(new - old) > self.threshold
            for old, new in zip(self.means, current)
        ]
```

### 39 · Model artifact integrity

```python
class ModelArtifact:
    def __init__(self, data):
        self.data = bytes(data)
        self.checksum = hashlib.sha256(self.data).hexdigest()

    def verify(self, candidate):
        candidate_checksum = hashlib.sha256(bytes(candidate)).hexdigest()
        return hmac.compare_digest(self.checksum, candidate_checksum)
```

### 40 · Versioned feature store

```python
class FeatureStore:
    def __init__(self):
        self._data = {}

    def put(self, entity, version, values):
        self._data[(entity, version)] = json.loads(json.dumps(values))

    def get(self, entity, version):
        return json.loads(json.dumps(self._data[(entity, version)]))
```

### 41 · Promote a model only when its metric improves

```python
class ModelCatalog:
    def __init__(self, initial_name, initial_score):
        self.champion = initial_name
        self.best_score = initial_score

    def consider(self, name, score):
        if score <= self.best_score:
            return False
        self.champion = name
        self.best_score = score
        return True
```

### 42 · Bounded least-recently-used inference cache

```python
class LRUCache:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._items = {}

    def get(self, key):
        if key not in self._items:
            return None
        value = self._items.pop(key)
        self._items[key] = value
        return value

    def put(self, key, value):
        self._items.pop(key, None)
        self._items[key] = value
        if len(self._items) > self.capacity:
            oldest = next(iter(self._items))
            del self._items[oldest]
```

### 43 · Discount strategies for a shopping cart

```python
class NoDiscount:
    def apply(self, subtotal):
        return subtotal

class PercentDiscount:
    def __init__(self, rate):
        if not 0 <= rate <= 1:
            raise ValueError("rate must be between 0 and 1")
        self.rate = rate

    def apply(self, subtotal):
        return subtotal * (1 - self.rate)

class ShoppingCart:
    def __init__(self, discount_strategy):
        self.discount_strategy = discount_strategy

    def total(self, items):
        subtotal = sum(items)
        return self.discount_strategy.apply(subtotal)
```

### 44 · Validated API pagination

```python
def paginate(items, page, page_size, max_page_size=100):
    if page < 1 or page_size < 1 or page_size > max_page_size:
        raise ValueError("page or page_size is outside the allowed range")
    start = (page - 1) * page_size
    return {
        "items": list(items[start:start + page_size]),
        "page": page,
        "page_size": page_size,
        "total": len(items),
    }
```

### 45 · Publish/subscribe event bus

```python
class EventBus:
    def __init__(self):
        self._handlers = {}

    def subscribe(self, event, handler):
        handlers = self._handlers.setdefault(event, [])
        handlers.append(handler)

        def unsubscribe():
            if handler in handlers:
                handlers.remove(handler)

        return unsubscribe

    def publish(self, event, payload):
        for handler in tuple(self._handlers.get(event, ())):
            handler(payload)
```

### 46 · Expiring in-memory cache

```python
class TTLCache:
    def __init__(self, clock):
        self.clock = clock
        self._items = {}

    def set(self, key, value, ttl):
        if ttl <= 0:
            raise ValueError("ttl must be positive")
        self._items[key] = (value, self.clock() + ttl)

    def get(self, key):
        entry = self._items.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if self.clock() >= expires_at:
            del self._items[key]
            return None
        return value
```

### 47 · Bounded retry policy

```python
class RetryPolicy:
    def __init__(self, max_attempts, retry_on):
        if max_attempts < 1:
            raise ValueError("max_attempts must be positive")
        self.max_attempts = max_attempts
        self.retry_on = retry_on

    def run(self, operation):
        for attempt in range(self.max_attempts):
            try:
                return operation()
            except self.retry_on:
                if attempt == self.max_attempts - 1:
                    raise
```

### 48 · Unit of work with rollback

```python
class UnitOfWork:
    def __init__(self, database):
        self.database = database

    def __enter__(self):
        self.database.begin()
        return self

    def __exit__(self, exception_type, exception, traceback):
        if exception_type is None:
            self.database.commit()
        else:
            self.database.rollback()
        return False
```

### 49 · Validated create-user request object

```python
class CreateUserRequest:
    def __init__(self, email, name):
        self.email = email
        self.name = name

    @classmethod
    def from_payload(cls, payload):
        if not isinstance(payload, dict) or set(payload) != {"email", "name"}:
            raise ValueError("payload must contain only email and name")
        email, name = payload["email"], payload["name"]
        if not isinstance(email, str) or not isinstance(name, str):
            raise ValueError("email and name must be strings")
        email, name = email.strip().lower(), name.strip()
        if not email or not name:
            raise ValueError("email and name are required")
        return cls(email, name)
```

### 50 · Order service with injected payment and repository

```python
class OrderService:
    def __init__(self, repository, payment_gateway):
        self.repository = repository
        self.payment_gateway = payment_gateway

    def place_order(self, user_id, items):
        if not items or any(price <= 0 for price in items):
            raise ValueError("order must contain positive-priced items")
        total = sum(items)
        if not self.payment_gateway.charge(user_id, total):
            raise ValueError("payment was declined")
        order = {"user_id": user_id, "items": list(items), "total": total}
        return self.repository.save(order)
```